In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="FR",
    max_iter=10000,
    tol=1e-100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0hello
0 10000
1 1e-100
, loss: 0.1672428995370865
epoch:  1hello
0 10000
1 1e-100
, loss: 0.3213324248790741
epoch:  2hello
0 10000
1 1e-100
, loss: 6.448299407958984
epoch:  3hello
0 10000
1 1e-100
, loss: 2885158.25
epoch:  4hello
0 10000
1 1e-100
, loss: inf
epoch:  5hello
0 10000
1 1e-100
, loss: nan
epoch:  6hello
0 10000
1 1e-100
, loss: nan
epoch:  7hello
0 10000
1 1e-100
, loss: nan
epoch:  8hello
0 10000
1 1e-100
, loss: nan
epoch:  9hello
0 10000
1 1e-100
, loss: nan
epoch:  10hello
0 10000
1 1e-100
, loss: nan
epoch:  11hello
0 10000
1 1e-100
, loss: nan
epoch:  12hello
0 10000
1 1e-100
, loss: nan
epoch:  13hello
0 10000
1 1e-100
, loss: nan
epoch:  14hello
0 10000
1 1e-100
, loss: nan
epoch:  15hello
0 10000
1 1e-100
, loss: nan
epoch:  16hello
0 10000
1 1e-100
, loss: nan
epoch:  17hello
0 10000
1 1e-100
, loss: nan
epoch:  18hello
0 10000
1 1e-100
, loss: nan
epoch:  19hello
0 10000
1 1e-100
, loss: nan
epoch:  20hello
0 10000
1 1e-100
, loss: nan
epoch:  21hell

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

ValueError: Input contains NaN.

In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3285638988018036
epoch:  1, loss: 0.17474116384983063
epoch:  2, loss: 0.11566995084285736
epoch:  3, loss: 0.08040527254343033
epoch:  4, loss: 0.06077496334910393
epoch:  5, loss: 0.04975054785609245
epoch:  6, loss: 0.04358642175793648
epoch:  7, loss: 0.040159910917282104
epoch:  8, loss: 0.03826329484581947
epoch:  9, loss: 0.03721582889556885
epoch:  10, loss: 0.03663703054189682
epoch:  11, loss: 0.0363156795501709
epoch:  12, loss: 0.03613552078604698
epoch:  13, loss: 0.03603179380297661
epoch:  14, loss: 0.03596944361925125
epoch:  15, loss: 0.035956982523202896
epoch:  16, loss: 0.0357891283929348
epoch:  17, loss: 0.03571801632642746
epoch:  18, loss: 0.03566041216254234
epoch:  19, loss: 0.035607364028692245
epoch:  20, loss: 0.035515885800123215
epoch:  21, loss: 0.03541332483291626
epoch:  22, loss: 0.035296183079481125
epoch:  23, loss: 0.03524204343557358
epoch:  24, loss: 0.03513984754681587
epoch:  25, loss: 0.03509344160556793
epoch:  26, loss: 0.

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7419114708900452
Test metrics:  R2 = 0.6246607899665833


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PRP+"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5210431814193726
epoch:  1, loss: 0.3409647047519684
epoch:  2, loss: 0.22930853068828583
epoch:  3, loss: 0.15860220789909363
epoch:  4, loss: 0.11378228664398193
epoch:  5, loss: 0.0853722095489502
epoch:  6, loss: 0.06737279146909714
epoch:  7, loss: 0.05597728490829468
epoch:  8, loss: 0.04876837134361267
epoch:  9, loss: 0.04421140253543854
epoch:  10, loss: 0.04133282229304314
epoch:  11, loss: 0.03951558470726013
epoch:  12, loss: 0.03836899250745773
epoch:  13, loss: 0.0376458615064621
epoch:  14, loss: 0.037189967930316925
epoch:  15, loss: 0.03690262883901596
epoch:  16, loss: 0.036721572279930115
epoch:  17, loss: 0.03660750389099121
epoch:  18, loss: 0.03653564304113388
epoch:  19, loss: 0.03649036958813667
epoch:  20, loss: 0.03646184504032135
epoch:  21, loss: 0.03644386678934097
epoch:  22, loss: 0.03643253073096275
epoch:  23, loss: 0.036425378173589706
epoch:  24, loss: 0.036420855671167374
epoch:  25, loss: 0.036417994648218155
epoch:  26, loss: 0.0

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7746648788452148
Test metrics:  R2 = 0.6730480194091797


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="HS"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3976782560348511
epoch:  1, loss: 0.21823042631149292
epoch:  2, loss: 0.044714316725730896
epoch:  3, loss: 0.04006190225481987
epoch:  4, loss: 0.036227550357580185
epoch:  5, loss: 0.036056261509656906
epoch:  6, loss: 0.03585159778594971
epoch:  7, loss: 0.03580942004919052
epoch:  8, loss: 0.03579053282737732
epoch:  9, loss: 0.03576125204563141
epoch:  10, loss: 0.03574901819229126
epoch:  11, loss: 0.035716712474823
epoch:  12, loss: 0.03563115373253822
epoch:  13, loss: 0.035583946853876114
epoch:  14, loss: 0.03553758189082146
epoch:  15, loss: 0.035451676696538925
epoch:  16, loss: 0.035322945564985275
epoch:  17, loss: 0.03526011481881142
epoch:  18, loss: 0.03518151864409447
epoch:  19, loss: 0.03494741767644882
epoch:  20, loss: 0.0348808579146862
epoch:  21, loss: 0.0348200649023056
epoch:  22, loss: 0.03472648933529854
epoch:  23, loss: 0.03451991826295853
epoch:  24, loss: 0.03404176980257034
epoch:  25, loss: 0.03394041210412979
epoch:  26, loss: 0.0

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7636985778808594
Test metrics:  R2 = 0.6484634280204773


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="DY"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1565745323896408
epoch:  1, loss: 0.0898767039179802
epoch:  2, loss: 0.0898767039179802
epoch:  3, loss: 0.037868600338697433
epoch:  4, loss: 0.037722550332546234
epoch:  5, loss: 0.037722546607255936
epoch:  6, loss: 0.03768780454993248
epoch:  7, loss: 0.037550222128629684
epoch:  8, loss: 0.037550222128629684
epoch:  9, loss: 0.03742041438817978
epoch:  10, loss: 0.03731083869934082
epoch:  11, loss: 0.03731083869934082
epoch:  12, loss: 0.03722041845321655
epoch:  13, loss: 0.037189047783613205
epoch:  14, loss: 0.037189047783613205
epoch:  15, loss: 0.037156298756599426
epoch:  16, loss: 0.03709058091044426
epoch:  17, loss: 0.03709058091044426
epoch:  18, loss: 0.03706969693303108
epoch:  19, loss: 0.037012938410043716
epoch:  20, loss: 0.037012938410043716
epoch:  21, loss: 0.03698434680700302
epoch:  22, loss: 0.036922916769981384
epoch:  23, loss: 0.036922916769981384
epoch:  24, loss: 0.036891840398311615
epoch:  25, loss: 0.03681834414601326
epoch:  26, 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8116456866264343
Test metrics:  R2 = 0.7160452604293823
